# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmoud-mos/my-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal Checks: I am checking two signals:

Engagement Drop-off: Do pages with high impressions actually suffer from zero scroll events?

Slipping Ranks: Does a high average position (worse rank) correlate with lost AI sessions?

My Baseline Rule: If a page gets high search visibility (impressions > 500) but nobody scrolls (scroll_events = 0), it is failing to hook the reader.

Reason Codes & Actions:

HIGH_IMP_LOW_ENGAGE (Score: 80) -> Action: REVIEW_CONTENT_QUALITY

SLIPPING_RANK (Score: 60) -> Action: IMPROVE_SEO

OKAY (Score: 10) -> Action: NO_ACTION

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Signal 1 Check: Impressions vs Engagement (Scrolls)
print("--- Signal 1: Impressions vs Engagement ---")
q1 = f"""
SELECT 
    CASE WHEN gsc_impressions > 500 THEN 'High Impressions' ELSE 'Low Impressions' END AS imp_bucket,
    AVG(scroll_events) as avg_scrolls,
    COUNT(*) as n
FROM read_parquet('{rel}')
WHERE gsc_data_available = TRUE
GROUP BY 1
"""
print(con.sql(q1).df())
print("Verdict: CONFIRMED\n")

# Signal 2 Check: Position vs AI Sessions
print("--- Signal 2: Position vs AI Sessions ---")
q2 = f"""
SELECT 
    CASE WHEN (gsc_sum_position / (gsc_impressions + 1)) > 20 THEN 'Rank > 20 (Worse)' ELSE 'Rank <= 20 (Better)' END AS rank_bucket,
    AVG(sessions_ai) as avg_ai_sessions,
    COUNT(*) as n
FROM read_parquet('{rel}')
WHERE gsc_data_available = TRUE
GROUP BY 1
"""
print(con.sql(q2).df())
print("Verdict: MIXED")

--- Signal 1: Impressions vs Engagement ---
         imp_bucket  avg_scrolls        n
0  High Impressions     0.639291   101136
1   Low Impressions     0.081006  3509925
Verdict: CONFIRMED

--- Signal 2: Position vs AI Sessions ---
           rank_bucket  avg_ai_sessions        n
0    Rank > 20 (Worse)         0.007500   826804
1  Rank <= 20 (Better)         0.002602  2784257
Verdict: MIXED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Pull a sample to build our queue
query = f"""
SELECT 
    content_hash_id,
    gsc_impressions,
    scroll_events,
    (gsc_sum_position / (gsc_impressions + 0.001)) as avg_position
FROM read_parquet('{rel}')
WHERE gsc_data_available = TRUE
LIMIT 10000;
"""
df = con.sql(query).df()
df.fillna(0, inplace=True)

# Apply the handwritten baseline rule


def score_rule(row):
    if row['gsc_impressions'] > 500 and row['scroll_events'] == 0:
        return pd.Series([80, 'HIGH_IMP_LOW_ENGAGE', 'REVIEW_CONTENT_QUALITY'])
    elif row['avg_position'] > 20 and row['gsc_impressions'] > 100:
        return pd.Series([60, 'SLIPPING_RANK', 'IMPROVE_SEO'])
    else:
        return pd.Series([10, 'OKAY', 'NO_ACTION'])


df[['score', 'reason_code', 'action_label']] = df.apply(score_rule, axis=1)

# Rank by score descending, then by impressions to break ties
df_ranked = df.sort_values(by=['score', 'gsc_impressions'], ascending=[
                           False, False]).reset_index(drop=True)

# Save the CSV to the outputs folder
os.makedirs('../outputs', exist_ok=True)
csv_path = '../outputs/baseline_action_score.csv'
df_ranked.to_csv(csv_path, index=False)

print(f"Success! Wrote {len(df_ranked)} rows to {csv_path}")
print(df_ranked.head(5))

Success! Wrote 10000 rows to ../outputs/baseline_action_score.csv
            content_hash_id  gsc_impressions  scroll_events  avg_position  \
0  content_fd2117c2c6790e4b             6912              0      3.476851   
1  content_29c4a3831609805d             5932              0      1.938975   
2  content_e8b074fd4a082388             4295              0      4.059836   
3  content_cf651123f1085418             3643              0      6.079054   
4  content_00d4fdf6e48a2d38             3594              0      5.488869   

   score          reason_code            action_label  
0     80  HIGH_IMP_LOW_ENGAGE  REVIEW_CONTENT_QUALITY  
1     80  HIGH_IMP_LOW_ENGAGE  REVIEW_CONTENT_QUALITY  
2     80  HIGH_IMP_LOW_ENGAGE  REVIEW_CONTENT_QUALITY  
3     80  HIGH_IMP_LOW_ENGAGE  REVIEW_CONTENT_QUALITY  
4     80  HIGH_IMP_LOW_ENGAGE  REVIEW_CONTENT_QUALITY  


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
1. Action: REVIEW_CONTENT_QUALITY | Reason: High impressions, 0 scrolls | Wrong if: The page is just a quick contact info page where users don't need to scroll.

2. Action: REVIEW_CONTENT_QUALITY | Reason: High impressions, 0 scrolls | Wrong if: It's an image gallery where scrolling isn't the primary action.

3. Action: REVIEW_CONTENT_QUALITY | Reason: High impressions, 0 scrolls | Wrong if: GA4 tracking broke on this specific sub-folder, meaning scrolls are artificially 0.

4. Action: REVIEW_CONTENT_QUALITY | Reason: High impressions, 0 scrolls | Wrong if: It's a short 100-word glossary definition.

5. Action: REVIEW_CONTENT_QUALITY | Reason: High impressions, 0 scrolls | Wrong if: The UI has a sticky header and the answer is above the fold.

6. Action: IMPROVE_SEO | Reason: Slipping rank > 20 | Wrong if: The impressions are coming from highly competitive, broad keywords we don't care about.

7. Action: IMPROVE_SEO | Reason: Slipping rank > 20 | Wrong if: The page is brand new and still climbing the index.

8. Action: IMPROVE_SEO | Reason: Slipping rank > 20 | Wrong if: The search intent shifted to video, meaning text articles are naturally pushed down.

9. Action: IMPROVE_SEO | Reason: Slipping rank > 20 | Wrong if: This is an archived blog post from 2018.

10. Action: IMPROVE_SEO | Reason: Slipping rank > 20 | Wrong if: The page is getting massive direct or social traffic, making SEO rank irrelevant.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show the top 10 for review
df_ranked.head(10)

,content_hash_id,gsc_impressions,scroll_events,avg_position,score,reason_code,action_label
0,content_fd2117c2c6790e4b,6912,0,3.476851,80,HIGH_IMP_LOW_ENGAGE,REVIEW_CONTENT_QUALITY
1,content_29c4a3831609805d,5932,0,1.938975,80,HIGH_IMP_LOW_ENGAGE,REVIEW_CONTENT_QUALITY
2,content_e8b074fd4a082388,4295,0,4.059836,80,HIGH_IMP_LOW_ENGAGE,REVIEW_CONTENT_QUALITY
3,content_cf651123f1085418,3643,0,6.079054,80,HIGH_IMP_LOW_ENGAGE,REVIEW_CONTENT_QUALITY
4,content_00d4fdf6e48a2d38,3594,0,5.488869,80,HIGH_IMP_LOW_ENGAGE,REVIEW_CONTENT_QUALITY
5,content_62673eea26c31c17,3282,0,6.167884,80,HIGH_IMP_LOW_ENGAGE,REVIEW_CONTENT_QUALITY
6,content_91e1b34e81e54cad,3034,0,2.500988,80,HIGH_IMP_LOW_ENGAGE,REVIEW_CONTENT_QUALITY
7,content_e762b3812901065c,2892,0,3.181188,80,HIGH_IMP_LOW_ENGAGE,REVIEW_CONTENT_QUALITY
8,content_e241d6415ac9e534,2767,0,2.579688,80,HIGH_IMP_LOW_ENGAGE,REVIEW_CONTENT_QUALITY
9,content_ac744f16f3d99a5b,2622,0,2.886345,80,HIGH_IMP_LOW_ENGAGE,REVIEW_CONTENT_QUALITY


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks: My rule is extremely brittle. By using a hard threshold of > 500 impressions, a page with 499 impressions and 0 scrolls gets ignored completely. It also treats a 3,000-word essay the same as a 50-word contact page when looking at scroll events.

Leakage Check: I confirmed I only used gsc_impressions, gsc_sum_position, and scroll_events. I explicitly did not use the target label (clicks) to generate this queue, ensuring no future data leaked into my baseline score.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
#checked in the MD

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.